In [ ]:
from src.auth import AuthenticationService
import ee
import geemap

In [ ]:
ok = AuthenticationService.authenticate(project_id="wide-office-411000")
if not ok:
    raise SystemExit("Could not initialize EE")


In [ ]:
# Carrega a coleção MODIS filtrando por data
dataset = ee.ImageCollection('MODIS/MOD09GA_006_NDVI') \
    .filter(ee.Filter.date('2018-04-01', '2018-06-01'))

# Seleciona a banda NDVI
colorized = dataset.select('NDVI')

# Definições de visualização
colorizedVis = {
    'min': 0,
    'max': 1,
    'palette': [
        'ffffff', 'ce7e45', 'df923d', 'f1b555', 'fcd163',
        '99b718', '74a901', '66a000', '529400', '3e8601',
        '207401', '056201', '004c00', '023b01', '012e01',
        '011d01', '011301'
    ],
}

# Cria o mapa interativo
Map = geemap.Map(center=[31.0529339857, -7.03125], zoom=2)

# Adiciona a camada colorida
Map.addLayer(colorized.mean(), colorizedVis, 'Colorized NDVI')

# Exibe o mapa
Map


In [ ]:
# Load the dataset
dataset = ee.ImageCollection('COPERNICUS/DEM/GLO30')
elevation = dataset.select('DEM')

# Visualization parameters
elevation_vis = {
    'min': 0.0,
    'max': 1000.0,
    'palette': ['0000ff', '00ffff', 'ffff00', 'ff0000', 'ffffff']
}

# Create a map
Map = geemap.Map()
Map.setCenter(-6.746, 46.529, 4)  # lon, lat, zoom
Map.addLayer(elevation, elevation_vis, 'DEM')

# Display map
Map


In [ ]:
dataset = ee.Image('CGIAR/SRTM90_V4')
elevation = dataset.select('elevation')
#slope = ee.Terrain.slope(elevation)

# Visualization parameters
elevation_vis = {
    'min': 0.0,
    'max': 1000.0,
    'palette': ['0000ff', '00ffff', 'ffff00', 'ff0000', 'ffffff']
}

# Create a map
Map = geemap.Map()
Map.setCenter(-6.746, 46.529, 4)  # lon, lat, zoom
Map.addLayer(elevation, elevation_vis, 'DEM')

# Display map
Map

In [ ]:
# Load the dataset and get the first image (2020 global map)
dataset = ee.ImageCollection('ESA/WorldCover/v200').first()

# Visualization parameters
visualization = {
    'bands': ['Map'],
}

# Create the map
Map = geemap.Map()
Map.centerObject(dataset)
Map.addLayer(dataset, visualization, 'Landcover')

# Display the map
Map


In [ ]:
# Load dataset
dataset = ee.Image('UMD/hansen/global_forest_change_2024_v1_12')

# Tree cover visualization (year 2000)
tree_cover_vis = {
    'bands': ['treecover2000'],
    'min': 0,
    'max': 100,
    'palette': ['black', 'green']
}

# Tree loss year visualization
tree_loss_vis = {
    'bands': ['lossyear'],
    'min': 0,
    'max': 24,
    'palette': ['yellow', 'red']
}

# Create interactive map
Map = geemap.Map()

# Add layers
Map.addLayer(dataset, tree_cover_vis, 'Tree Cover 2000')
Map.addLayer(dataset, tree_loss_vis, 'Tree Loss Year')

# Set initial view (global)
Map.setCenter(0, 0, 2)

# Display map
Map


In [ ]:
# Load the WorldClim bioclimatic dataset
dataset = ee.Image("WORLDCLIM/V1/BIO")

# Select 'bio01' (Annual Mean Temperature) and apply scale factor 0.1
annual_mean_temperature = dataset.select('bio01').multiply(0.1)

# Visualization parameters
vis_params = {
    'min': -23,
    'max': 30,
    'palette': ['blue', 'purple', 'cyan', 'green', 'yellow', 'red']
}

# Define the map center
center_coords = [52.4, 71.7]  # latitude, longitude
zoom_level = 3

# Use geemap to display in Python
import geemap
Map = geemap.Map(center=center_coords, zoom=zoom_level)
Map.addLayer(annual_mean_temperature, vis_params, 'Annual Mean Temperature')
Map


In [ ]:
dataset = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate(
    '2022-01-01', '2022-02-01'
)


# Applies scaling factors.
def apply_scale_factors(image):
  optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
  thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
  return image.addBands(optical_bands, None, True).addBands(
      thermal_bands, None, True
  )


dataset = dataset.map(apply_scale_factors)

visualization = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 0.0,
    'max': 0.3,
}

m = geemap.Map()
m.set_center(-114.2579, 38.9275, 8)
m.add_layer(dataset, visualization, 'True Color (432)')
m

In [ ]:
def mask_s2_clouds(image):
  """Masks clouds in a Sentinel-2 image using the QA band.

  Args:
      image (ee.Image): A Sentinel-2 image.

  Returns:
      ee.Image: A cloud-masked Sentinel-2 image.
  """
  qa = image.select('QA60')

  # Bits 10 and 11 are clouds and cirrus, respectively.
  cloud_bit_mask = 1 << 10
  cirrus_bit_mask = 1 << 11

  # Both flags should be set to zero, indicating clear conditions.
  mask = (
      qa.bitwiseAnd(cloud_bit_mask)
      .eq(0)
      .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
  )

  return image.updateMask(mask).divide(10000)


dataset = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2020-01-01', '2020-01-30')
    # Pre-filter to get less cloudy granules.
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds)
)

visualization = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2'],
}

m = geemap.Map()
m.set_center(83.277, 17.7009, 12)
m.add_layer(dataset.mean(), visualization, 'RGB')
m